# 🤖 Reinforcement Learning Maze Solver (8x8)

A complete, self-contained Colab project. An RL agent (PPO via Stable-Baselines3) learns to solve
**randomly generated 8x8 mazes** purely from rewards/punishments — no hardcoded solving logic.

**Pipeline:** Maze generation → Gymnasium environment → PPO training → Testing on unseen mazes →
Visualization/animation → Model saving/loading.

Run all cells top-to-bottom. If `maze_agent.zip` already exists, training is skipped automatically
(set `Config.FORCE_RETRAIN = True` to override).


In [ ]:
# Install dependencies (safe to re-run)
!pip install -q gymnasium stable-baselines3 torch numpy matplotlib


In [ ]:
# Imports
import os
import time
import random
import json
import collections
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display, clear_output

import torch
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 1. `Config` — all settings live here (no magic numbers elsewhere)

In [ ]:
class Config:
    """
    Central configuration for the whole project.
    Every other class reads its parameters from here so nothing is hardcoded
    inline in the environment / training / testing code.
    """

    # ---- Maze geometry ----
    GRID_ROWS = 8
    GRID_COLS = 8
    WALL_DENSITY = 0.25          # fraction of non S/G cells that become walls (bonus: random wall density)
    MIN_WALL_DENSITY = 0.15      # used for curriculum learning (easy mazes)
    MAX_WALL_DENSITY = 0.35      # used for curriculum learning (hard mazes)

    # ---- Rewards (Reward System spec) ----
    REWARD_GOAL = 100.0
    REWARD_WALL = -10.0
    REWARD_STEP = -1.0
    REWARD_OUT_OF_BOUNDS = -15.0
    REWARD_REPEAT_POSITION = -3.0

    # ---- Episode rules ----
    MAX_STEPS_PER_EPISODE = 128
    N_ACTIONS = 4                 # up, down, left, right

    # ---- Symbols for human-readable rendering ----
    SYMBOL_EMPTY = "⬜"
    SYMBOL_WALL = "⬛"
    SYMBOL_START = "🟢"
    SYMBOL_GOAL = "🔴"
    SYMBOL_AGENT = "🤖"

    # ---- Training ----
    ALGORITHM = "PPO"              # "PPO" (preferred) or "DQN" (fallback)
    TOTAL_EPISODES = 100_000       # spec requirement: train for >= 100,000 episodes
    # PPO in SB3 is configured in *timesteps*, not episodes. We estimate timesteps
    # from episodes * average-steps-per-episode so the spec's "episodes" requirement is honored.
    AVG_STEPS_ESTIMATE = 40
    TOTAL_TIMESTEPS = TOTAL_EPISODES * AVG_STEPS_ESTIMATE

    LEARNING_RATE = 3e-4
    N_STEPS = 1024          # PPO rollout length
    BATCH_SIZE = 256
    N_EPOCHS = 10
    GAMMA = 0.99
    GAE_LAMBDA = 0.95
    CLIP_RANGE = 0.2
    ENT_COEF = 0.01

    PROGRESS_LOG_INTERVAL = 100     # log every 100 episodes
    EVAL_INTERVAL_EPISODES = 500    # bonus: evaluation every 500 episodes
    EARLY_STOP_SUCCESS_RATE = 0.97  # bonus: early stopping threshold
    EARLY_STOP_PATIENCE_EVALS = 10  # how many consecutive good evals before stopping

    # ---- Curriculum learning (bonus) ----
    USE_CURRICULUM = True
    CURRICULUM_STAGES = [
        # (episode_fraction_at_which_stage_starts, wall_density)
        (0.0, 0.15),
        (0.25, 0.20),
        (0.5, 0.25),
        (0.75, 0.30),
        (0.9, 0.35),
    ]

    # ---- Testing ----
    N_TEST_MAZES = 20

    # ---- Visualization ----
    ANIMATION_DELAY_SEC = 0.2

    # ---- Files / paths ----
    MODEL_DIR = "/content/maze_rl_artifacts"
    MODEL_PATH = os.path.join(MODEL_DIR, "maze_agent.zip")
    BEST_MODEL_DIR = os.path.join(MODEL_DIR, "best_model")
    REWARDS_LOG_PATH = os.path.join(MODEL_DIR, "training_rewards.json")
    GRAPH_REWARD_PATH = os.path.join(MODEL_DIR, "graph_reward.png")
    GRAPH_SUCCESS_PATH = os.path.join(MODEL_DIR, "graph_success.png")
    GRAPH_STEPS_PATH = os.path.join(MODEL_DIR, "graph_steps.png")
    GRAPH_MOVING_AVG_PATH = os.path.join(MODEL_DIR, "graph_moving_avg.png")
    TENSORBOARD_DIR = os.path.join(MODEL_DIR, "tensorboard")

    # ---- Misc ----
    FORCE_RETRAIN = False     # set True to retrain even if a saved model exists
    RANDOM_SEED = None        # set an int for reproducibility, or None for full randomness
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    @staticmethod
    def setup_dirs():
        """Create all output directories if they do not already exist."""
        os.makedirs(Config.MODEL_DIR, exist_ok=True)
        os.makedirs(Config.BEST_MODEL_DIR, exist_ok=True)
        os.makedirs(Config.TENSORBOARD_DIR, exist_ok=True)

    @staticmethod
    def set_seed(seed):
        """Apply a global random seed (bonus: random seed option) for reproducibility."""
        Config.RANDOM_SEED = seed
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

Config.setup_dirs()
print("Config ready. Device:", Config.DEVICE)


## 2. `MazeGenerator` — produces a fresh, solvable random maze every call

In [ ]:
class MazeGenerator:
    """
    Generates random 8x8 mazes made of 0 (empty) and 1 (wall) cells, plus a
    Start (S) and Goal (G) position. Guarantees solvability via BFS validation:
    if a randomly generated maze has no path from S to G, it is regenerated.
    """

    def __init__(self, rows=None, cols=None, wall_density=None):
        self.rows = rows or Config.GRID_ROWS
        self.cols = cols or Config.GRID_COLS
        self.wall_density = wall_density if wall_density is not None else Config.WALL_DENSITY

    def _random_layout(self):
        """Create a random grid of 0/1 values plus random S and G corners-ish positions."""
        grid = np.zeros((self.rows, self.cols), dtype=np.int8)

        # Randomly place walls according to wall_density, never on S or G (set after).
        total_cells = self.rows * self.cols
        n_walls = int(total_cells * self.wall_density)
        wall_positions = random.sample(range(total_cells), n_walls)
        for pos in wall_positions:
            r, c = divmod(pos, self.cols)
            grid[r, c] = 1

        # Pick start and goal at random distinct empty-ish cells (force them open).
        start = (random.randint(0, self.rows - 1), random.randint(0, self.cols - 1))
        goal = start
        while goal == start:
            goal = (random.randint(0, self.rows - 1), random.randint(0, self.cols - 1))

        grid[start] = 0
        grid[goal] = 0
        return grid, start, goal

    def _bfs_reachable(self, grid, start, goal):
        """Breadth-first search to check whether goal is reachable from start."""
        rows, cols = grid.shape
        visited = np.zeros_like(grid, dtype=bool)
        queue = deque([start])
        visited[start] = True
        while queue:
            r, c = queue.popleft()
            if (r, c) == goal:
                return True
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and not visited[nr, nc] and grid[nr, nc] == 0:
                    visited[nr, nc] = True
                    queue.append((nr, nc))
        return False

    def generate(self):
        """
        Keep generating random layouts until one is solvable (S can reach G).
        Returns: (grid: np.ndarray[int8], start: (r,c), goal: (r,c))
        """
        while True:
            grid, start, goal = self._random_layout()
            if self._bfs_reachable(grid, start, goal):
                return grid, start, goal

    def render_grid_with_symbols(self, grid, start, goal, agent_pos=None):
        """
        Build a list-of-strings human-readable rendering of the maze using the
        emoji symbols defined in Config. agent_pos overrides start's symbol if given.
        """
        rows, cols = grid.shape
        lines = []
        for r in range(rows):
            row_syms = []
            for c in range(cols):
                pos = (r, c)
                if agent_pos is not None and pos == agent_pos:
                    row_syms.append(Config.SYMBOL_AGENT)
                elif pos == goal:
                    row_syms.append(Config.SYMBOL_GOAL)
                elif pos == start:
                    row_syms.append(Config.SYMBOL_START)
                elif grid[r, c] == 1:
                    row_syms.append(Config.SYMBOL_WALL)
                else:
                    row_syms.append(Config.SYMBOL_EMPTY)
            lines.append("".join(row_syms))
        return lines

# Quick sanity check
_gen = MazeGenerator()
_grid, _start, _goal = _gen.generate()
for line in _gen.render_grid_with_symbols(_grid, _start, _goal):
    print(line)
print("Start:", _start, "Goal:", _goal)


## 3. `RewardSystem` — centralizes all reward logic (spec: Reward System)

In [ ]:
class RewardSystem:
    """
    Encapsulates the reward rules so the environment stays clean. The agent
    learns ONLY through these signals — no maze solution is ever hardcoded.
    """

    def __init__(self):
        self.r_goal = Config.REWARD_GOAL
        self.r_wall = Config.REWARD_WALL
        self.r_step = Config.REWARD_STEP
        self.r_out = Config.REWARD_OUT_OF_BOUNDS
        self.r_repeat = Config.REWARD_REPEAT_POSITION

    def compute(self, event):
        """
        event is one of: "goal", "wall", "out_of_bounds", "repeat", "step"
        Returns the scalar reward for that event.
        """
        return {
            "goal": self.r_goal,
            "wall": self.r_wall,
            "out_of_bounds": self.r_out,
            "repeat": self.r_repeat,
            "step": self.r_step,
        }[event]


## 4. `MazeEnvironment` — a Gymnasium env wrapping generator + reward system

In [ ]:
class MazeEnvironment(gym.Env):
    """
    Gymnasium-compatible environment for the maze-solving task.

    Observation: a flattened vector containing
        - the full maze layout (rows*cols binary wall map)
        - the agent's current (row, col), normalized
        - the goal's (row, col), normalized
    This gives PPO/DQN everything needed to learn navigation without any
    explicit path information.

    Action space: Discrete(4) -> 0=Up, 1=Down, 2=Left, 3=Right.

    A brand-new random (but always solvable) maze is generated every reset(),
    so the agent never memorizes a single fixed maze.
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, wall_density=None, fixed_maze=None):
        super().__init__()
        self.rows = Config.GRID_ROWS
        self.cols = Config.GRID_COLS
        self.max_steps = Config.MAX_STEPS_PER_EPISODE
        self.wall_density = wall_density if wall_density is not None else Config.WALL_DENSITY
        self.generator = MazeGenerator(self.rows, self.cols, self.wall_density)
        self.reward_system = RewardSystem()
        self.fixed_maze = fixed_maze  # optional (grid, start, goal) to force a specific maze (used in testing)

        # Observation: maze grid flattened + agent pos (2) + goal pos (2), all float32
        obs_len = self.rows * self.cols + 2 + 2
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(obs_len,), dtype=np.float32)
        self.action_space = spaces.Discrete(Config.N_ACTIONS)

        # Action -> (delta_row, delta_col)
        self._action_deltas = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}

        self.grid = None
        self.start = None
        self.goal = None
        self.agent_pos = None
        self.steps_taken = 0
        self.visited_positions = None
        self.episode_path = []  # records the path for visualization/testing

    def set_wall_density(self, density):
        """Allows curriculum learning to ramp difficulty up over training."""
        self.wall_density = density
        self.generator = MazeGenerator(self.rows, self.cols, density)

    def _new_maze(self):
        """Generate a new maze (or use a fixed one, e.g. for testing on held-out mazes)."""
        if self.fixed_maze is not None:
            grid, start, goal = self.fixed_maze
            return grid.copy(), start, goal
        return self.generator.generate()

    def _get_obs(self):
        """Build the flattened observation vector described above."""
        flat_grid = self.grid.flatten().astype(np.float32)
        agent_norm = np.array([self.agent_pos[0] / (self.rows - 1), self.agent_pos[1] / (self.cols - 1)], dtype=np.float32)
        goal_norm = np.array([self.goal[0] / (self.rows - 1), self.goal[1] / (self.cols - 1)], dtype=np.float32)
        return np.concatenate([flat_grid, agent_norm, goal_norm])

    def reset(self, seed=None, options=None):
        """Start a new episode with a brand-new random maze."""
        super().reset(seed=seed)
        self.grid, self.start, self.goal = self._new_maze()
        self.agent_pos = self.start
        self.steps_taken = 0
        self.visited_positions = {self.agent_pos}
        self.episode_path = [self.agent_pos]
        return self._get_obs(), {"start": self.start, "goal": self.goal}

    def step(self, action):
        """
        Apply one action. Returns (obs, reward, terminated, truncated, info)
        following the Reward System spec exactly.
        """
        dr, dc = self._action_deltas[int(action)]
        nr, nc = self.agent_pos[0] + dr, self.agent_pos[1] + dc
        self.steps_taken += 1

        terminated = False
        truncated = False

        # 1) Moving outside the grid boundary
        if not (0 <= nr < self.rows and 0 <= nc < self.cols):
            reward = self.reward_system.compute("out_of_bounds")
            # agent position does not change
        else:
            # 2) Hitting a wall cell
            if self.grid[nr, nc] == 1:
                reward = self.reward_system.compute("wall")
                # agent position does not change (bounced back)
            else:
                new_pos = (nr, nc)
                if new_pos == self.goal:
                    self.agent_pos = new_pos
                    reward = self.reward_system.compute("goal")
                    terminated = True
                elif new_pos in self.visited_positions:
                    # 3) Re-visiting an already-visited cell
                    self.agent_pos = new_pos
                    reward = self.reward_system.compute("repeat")
                else:
                    # 4) Normal move into a new empty cell
                    self.agent_pos = new_pos
                    reward = self.reward_system.compute("step")

        self.visited_positions.add(self.agent_pos)
        self.episode_path.append(self.agent_pos)

        # 5) Step limit reached -> episode truncated
        if not terminated and self.steps_taken >= self.max_steps:
            truncated = True

        info = {
            "agent_pos": self.agent_pos,
            "goal": self.goal,
            "success": terminated,
            "steps": self.steps_taken,
        }
        return self._get_obs(), reward, terminated, truncated, info

    def render_text(self):
        """Return the emoji-rendered maze with the agent at its current position."""
        return self.generator.render_grid_with_symbols(self.grid, self.start, self.goal, agent_pos=self.agent_pos)


## 5. `MazeAgent` — wraps the Stable-Baselines3 model (PPO, fallback DQN)

In [ ]:
class MazeAgent:
    """
    Thin wrapper around a Stable-Baselines3 model. Builds either a PPO agent
    (preferred per spec) or, if requested, a DQN agent as fallback.
    Also exposes a single-step "act" method used during testing/visualization.
    """

    def __init__(self, env, algorithm=None):
        self.algorithm = algorithm or Config.ALGORITHM
        self.env = env
        self.model = self._build_model()

    def _build_model(self):
        """Construct the SB3 model with hyperparameters pulled from Config."""
        common_kwargs = dict(
            policy="MlpPolicy",
            env=self.env,
            verbose=0,
            device=Config.DEVICE,
            tensorboard_log=Config.TENSORBOARD_DIR,
            seed=Config.RANDOM_SEED,
        )
        if self.algorithm == "PPO":
            return PPO(
                learning_rate=Config.LEARNING_RATE,
                n_steps=Config.N_STEPS,
                batch_size=Config.BATCH_SIZE,
                n_epochs=Config.N_EPOCHS,
                gamma=Config.GAMMA,
                gae_lambda=Config.GAE_LAMBDA,
                clip_range=Config.CLIP_RANGE,
                ent_coef=Config.ENT_COEF,
                **common_kwargs,
            )
        else:  # fallback DQN
            return DQN(
                learning_rate=Config.LEARNING_RATE,
                gamma=Config.GAMMA,
                **common_kwargs,
            )

    def act(self, obs, deterministic=True):
        """Pick an action for a single observation (used during testing)."""
        action, _ = self.model.predict(obs, deterministic=deterministic)
        return int(action)

    def save(self, path):
        self.model.save(path)

    def load(self, path):
        """Load weights into a fresh model of the same algorithm/env."""
        cls = PPO if self.algorithm == "PPO" else DQN
        self.model = cls.load(path, env=self.env, device=Config.DEVICE)
        return self.model


## 6. `Trainer` — runs training, logs progress, handles curriculum + early stopping
Implements: progress logging every 100 episodes, evaluation every 500 episodes,
curriculum learning (ramping wall density), early stopping, and best-model saving.


In [ ]:
class ProgressCallback(BaseCallback):
    """
    SB3 callback that:
      - tracks episode rewards/lengths/successes as episodes complete
      - logs progress every Config.PROGRESS_LOG_INTERVAL episodes
      - applies curriculum learning by adjusting env wall density over time
      - runs periodic evaluation and tracks the best success rate (best-model saving)
      - signals early stopping once success rate is consistently high
    """

    def __init__(self, env, total_episode_target, verbose=0):
        super().__init__(verbose)
        self.env = env
        self.total_episode_target = total_episode_target
        self.episode_count = 0
        self.history = {"episode": [], "avg_reward": [], "success_rate": [], "avg_steps": [], "loss": []}
        self._reward_window = deque(maxlen=Config.PROGRESS_LOG_INTERVAL)
        self._success_window = deque(maxlen=Config.PROGRESS_LOG_INTERVAL)
        self._steps_window = deque(maxlen=Config.PROGRESS_LOG_INTERVAL)
        self._best_success_rate = -1.0
        self._consecutive_good_evals = 0
        self.stop_training_flag = False

    def _current_wall_density(self):
        """Curriculum learning: pick wall density based on training progress fraction."""
        frac = self.episode_count / max(1, self.total_episode_target)
        density = Config.CURRICULUM_STAGES[0][1]
        for stage_frac, stage_density in Config.CURRICULUM_STAGES:
            if frac >= stage_frac:
                density = stage_density
        return density

    def _on_step(self):
        # SB3 VecEnv exposes "dones" and "infos" in self.locals
        dones = self.locals.get("dones")
        infos = self.locals.get("infos")
        rewards = self.locals.get("rewards")
        if dones is not None:
            for i, done in enumerate(dones):
                if done:
                    self.episode_count += 1
                    info = infos[i]
                    ep_reward = info.get("episode", {}).get("r") if "episode" in info else None
                    ep_len = info.get("episode", {}).get("l") if "episode" in info else info.get("steps")
                    success = bool(info.get("success", False))
                    if ep_reward is None:
                        ep_reward = rewards[i]
                    self._reward_window.append(ep_reward)
                    self._success_window.append(1.0 if success else 0.0)
                    self._steps_window.append(ep_len if ep_len is not None else 0)

                    # Curriculum: ramp wall density on the underlying envs
                    if Config.USE_CURRICULUM:
                        new_density = self._current_wall_density()
                        try:
                            self.training_env.env_method("set_wall_density", new_density)
                        except Exception:
                            pass

                    if self.episode_count % Config.PROGRESS_LOG_INTERVAL == 0:
                        avg_r = float(np.mean(self._reward_window)) if self._reward_window else 0.0
                        succ = float(np.mean(self._success_window)) if self._success_window else 0.0
                        avg_s = float(np.mean(self._steps_window)) if self._steps_window else 0.0
                        loss = None
                        try:
                            loss = float(self.model.logger.name_to_value.get("train/loss", np.nan))
                        except Exception:
                            loss = float("nan")
                        self.history["episode"].append(self.episode_count)
                        self.history["avg_reward"].append(avg_r)
                        self.history["success_rate"].append(succ)
                        self.history["avg_steps"].append(avg_s)
                        self.history["loss"].append(loss)
                        print(f"Episode {self.episode_count:>7} | "
                              f"AvgReward {avg_r:8.2f} | SuccessRate {succ*100:6.2f}% | "
                              f"AvgSteps {avg_s:6.2f} | Loss {loss:.4f}" if not np.isnan(loss) else
                              f"Episode {self.episode_count:>7} | AvgReward {avg_r:8.2f} | "
                              f"SuccessRate {succ*100:6.2f}% | AvgSteps {avg_s:6.2f} | Loss n/a")

                    # Evaluation + early stopping every EVAL_INTERVAL_EPISODES
                    if self.episode_count % Config.EVAL_INTERVAL_EPISODES == 0:
                        succ = float(np.mean(self._success_window)) if self._success_window else 0.0
                        print(f"  >> Eval checkpoint @ episode {self.episode_count}: success_rate={succ*100:.2f}%")
                        if succ > self._best_success_rate:
                            self._best_success_rate = succ
                            self.model.save(os.path.join(Config.BEST_MODEL_DIR, "best_model.zip"))
                            print(f"  >> New best model saved (success_rate={succ*100:.2f}%)")
                        if succ >= Config.EARLY_STOP_SUCCESS_RATE:
                            self._consecutive_good_evals += 1
                        else:
                            self._consecutive_good_evals = 0
                        if self._consecutive_good_evals >= Config.EARLY_STOP_PATIENCE_EVALS:
                            print("  >> Early stopping criterion met. Stopping training.")
                            self.stop_training_flag = True

        if self.episode_count >= self.total_episode_target or self.stop_training_flag:
            return False  # stop training
        return True


class Trainer:
    """
    Orchestrates the full training run: builds the vectorized environment,
    builds the agent, attaches the ProgressCallback, and trains for the
    configured number of timesteps/episodes.
    """

    def __init__(self, config=Config):
        self.config = config
        raw_env = MazeEnvironment()
        self.monitored_env = Monitor(raw_env)
        self.vec_env = DummyVecEnv([lambda: self.monitored_env])
        self.agent = MazeAgent(self.vec_env, algorithm=config.ALGORITHM)
        self.callback = ProgressCallback(self.vec_env, total_episode_target=config.TOTAL_EPISODES)

    def train(self):
        print(f"Starting training with {self.config.ALGORITHM} for up to "
              f"{self.config.TOTAL_EPISODES} episodes (~{self.config.TOTAL_TIMESTEPS} timesteps cap)...")
        start_time = time.time()
        self.agent.model.learn(
            total_timesteps=self.config.TOTAL_TIMESTEPS,
            callback=self.callback,
            progress_bar=False,
        )
        elapsed = time.time() - start_time
        print(f"Training finished in {elapsed/60:.2f} minutes "
              f"after {self.callback.episode_count} episodes.")
        return self.callback.history

    def save_everything(self):
        """Save the trained model + the raw reward/success/steps history to disk."""
        self.agent.save(self.config.MODEL_PATH)
        with open(self.config.REWARDS_LOG_PATH, "w") as f:
            json.dump(self.callback.history, f)
        print(f"Model saved to {self.config.MODEL_PATH}")
        print(f"Training history saved to {self.config.REWARDS_LOG_PATH}")


## 7. `ModelManager` — load existing model automatically, or train a new one

In [ ]:
class ModelManager:
    """
    Handles the "load if exists, else train" logic, plus training-resume support.
    """

    def __init__(self, config=Config):
        self.config = config

    def model_exists(self):
        return os.path.exists(self.config.MODEL_PATH)

    def get_or_train(self, force_retrain=None):
        """
        If a saved model exists and force_retrain is False, load it.
        Otherwise, run a full Trainer session and save the result.
        Returns (agent, history_or_None, trainer_or_None)
        """
        force_retrain = self.config.FORCE_RETRAIN if force_retrain is None else force_retrain

        if self.model_exists() and not force_retrain:
            print(f"Found existing model at {self.config.MODEL_PATH} -> loading (no retraining).")
            raw_env = MazeEnvironment()
            vec_env = DummyVecEnv([lambda: Monitor(raw_env)])
            agent = MazeAgent(vec_env, algorithm=self.config.ALGORITHM)
            agent.load(self.config.MODEL_PATH)
            history = None
            if os.path.exists(self.config.REWARDS_LOG_PATH):
                with open(self.config.REWARDS_LOG_PATH) as f:
                    history = json.load(f)
            return agent, history, None
        else:
            print("No existing model found (or retraining forced) -> training from scratch.")
            trainer = Trainer(self.config)
            history = trainer.train()
            trainer.save_everything()
            return trainer.agent, history, trainer

    def resume_training(self, extra_timesteps):
        """Bonus: training-resume support — continue learning on top of a saved model."""
        if not self.model_exists():
            raise FileNotFoundError("No saved model to resume from.")
        raw_env = MazeEnvironment()
        vec_env = DummyVecEnv([lambda: Monitor(raw_env)])
        agent = MazeAgent(vec_env, algorithm=self.config.ALGORITHM)
        agent.load(self.config.MODEL_PATH)
        callback = ProgressCallback(vec_env, total_episode_target=10**9)  # no hard episode cap on resume
        agent.model.learn(total_timesteps=extra_timesteps, callback=callback, reset_num_timesteps=False)
        agent.save(self.config.MODEL_PATH)
        print("Resumed training complete; model re-saved.")
        return agent, callback.history


## 8. `Tester` — evaluates the trained agent on brand-new, unseen mazes

In [ ]:
class Tester:
    """
    Generates N completely new random mazes (never seen in training) and runs
    the trained agent on each, recording path/steps/success/reward.
    """

    def __init__(self, agent, config=Config):
        self.agent = agent
        self.config = config
        self.generator = MazeGenerator()

    def run_single_test(self, grid=None, start=None, goal=None, render=True):
        """Run the agent on one maze (generates a new one if not supplied)."""
        if grid is None:
            grid, start, goal = self.generator.generate()
        env = MazeEnvironment(fixed_maze=(grid, start, goal))
        obs, _ = env.reset()
        total_reward = 0.0
        success = False
        for _ in range(self.config.MAX_STEPS_PER_EPISODE):
            action = self.agent.act(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            if terminated:
                success = True
                break
            if truncated:
                break

        result = {
            "grid": grid,
            "start": start,
            "goal": goal,
            "path": env.episode_path,
            "steps": env.steps_taken,
            "success": success,
            "reward": total_reward,
        }
        if render:
            self._print_result(result)
        return result

    def _print_result(self, result):
        print("Maze:")
        gen = MazeGenerator()
        for line in gen.render_grid_with_symbols(result["grid"], result["start"], result["goal"]):
            print(line)
        print(f"Path taken ({len(result['path'])} positions): {result['path']}")
        print(f"Steps: {result['steps']}")
        print(f"Result: {'SUCCESS' if result['success'] else 'FAILURE'}")
        print(f"Total reward: {result['reward']:.2f}")
        print("-" * 60)

    def run_test_suite(self, n_mazes=None):
        """Run the agent on n_mazes brand-new mazes and summarize results."""
        n_mazes = n_mazes or self.config.N_TEST_MAZES
        results = []
        for i in range(n_mazes):
            print(f"=== Test maze {i+1}/{n_mazes} ===")
            results.append(self.run_single_test(render=True))

        successes = sum(r["success"] for r in results)
        avg_steps = np.mean([r["steps"] for r in results])
        avg_reward = np.mean([r["reward"] for r in results])
        print("=" * 60)
        print(f"TEST SUMMARY over {n_mazes} unseen mazes:")
        print(f"  Success rate: {successes}/{n_mazes} ({successes/n_mazes*100:.1f}%)")
        print(f"  Average steps: {avg_steps:.2f}")
        print(f"  Average reward: {avg_reward:.2f}")
        return results


## 9. `Visualizer` — matplotlib animation of the agent solving a maze + training graphs

In [ ]:
class Visualizer:
    """
    Handles:
      - animated playback of one solved (or attempted) maze using matplotlib.animation
      - static training graphs: reward, success rate, steps, moving-average reward
    """

    def __init__(self, config=Config):
        self.config = config

    def animate_episode(self, result, episode_label="Final Demo"):
        """
        Animate the recorded path on the maze grid. Shows current episode label,
        cumulative reward-so-far, and current step in the title, updated each frame.
        Delay between frames follows Config.ANIMATION_DELAY_SEC.
        """
        grid, start, goal, path = result["grid"], result["start"], result["goal"], result["path"]
        rows, cols = grid.shape

        fig, ax = plt.subplots(figsize=(5, 5))

        def draw_frame(step_idx):
            ax.clear()
            ax.set_xlim(-0.5, cols - 0.5)
            ax.set_ylim(-0.5, rows - 0.5)
            ax.invert_yaxis()
            ax.set_xticks([])
            ax.set_yticks([])

            for r in range(rows):
                for c in range(cols):
                    color = "black" if grid[r, c] == 1 else "white"
                    ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color=color, ec="gray"))

            ax.add_patch(plt.Rectangle((start[1] - 0.5, start[0] - 0.5), 1, 1, color="green", alpha=0.5))
            ax.add_patch(plt.Rectangle((goal[1] - 0.5, goal[0] - 0.5), 1, 1, color="red", alpha=0.5))

            agent_r, agent_c = path[step_idx]
            ax.plot(agent_c, agent_r, "o", color="blue", markersize=18)

            cumulative_steps = step_idx
            ax.set_title(f"{episode_label} | Step {cumulative_steps}/{len(path)-1}")
            return ax

        def update(frame):
            return draw_frame(frame)

        anim = animation.FuncAnimation(
            fig, update, frames=len(path), interval=self.config.ANIMATION_DELAY_SEC * 1000, repeat=False
        )
        plt.close(fig)
        return HTML(anim.to_jshtml())

    def plot_training_graphs(self, history):
        """
        Plot Episode vs Reward, Episode vs Success Rate, Episode vs Steps, and
        Episode vs Moving-Average Reward. Saves each figure to disk per Config paths.
        """
        if not history or not history.get("episode"):
            print("No training history available to plot (model was loaded, not trained).")
            return

        episodes = history["episode"]
        rewards = history["avg_reward"]
        successes = history["success_rate"]
        steps = history["avg_steps"]

        # Episode vs Reward
        plt.figure(figsize=(8, 4))
        plt.plot(episodes, rewards, color="tab:blue")
        plt.xlabel("Episode"); plt.ylabel("Average Reward"); plt.title("Episode vs Reward")
        plt.grid(True); plt.tight_layout()
        plt.savefig(self.config.GRAPH_REWARD_PATH); plt.show()

        # Episode vs Success Rate
        plt.figure(figsize=(8, 4))
        plt.plot(episodes, [s * 100 for s in successes], color="tab:green")
        plt.xlabel("Episode"); plt.ylabel("Success Rate (%)"); plt.title("Episode vs Success Rate")
        plt.grid(True); plt.tight_layout()
        plt.savefig(self.config.GRAPH_SUCCESS_PATH); plt.show()

        # Episode vs Steps
        plt.figure(figsize=(8, 4))
        plt.plot(episodes, steps, color="tab:orange")
        plt.xlabel("Episode"); plt.ylabel("Average Steps"); plt.title("Episode vs Steps")
        plt.grid(True); plt.tight_layout()
        plt.savefig(self.config.GRAPH_STEPS_PATH); plt.show()

        # Episode vs Moving Average Reward (window of 10 logged points)
        window = 10
        moving_avg = np.convolve(rewards, np.ones(window) / window, mode="valid") if len(rewards) >= window else rewards
        ma_episodes = episodes[window - 1:] if len(rewards) >= window else episodes
        plt.figure(figsize=(8, 4))
        plt.plot(ma_episodes, moving_avg, color="tab:purple")
        plt.xlabel("Episode"); plt.ylabel("Moving Avg Reward"); plt.title("Episode vs Moving Average Reward")
        plt.grid(True); plt.tight_layout()
        plt.savefig(self.config.GRAPH_MOVING_AVG_PATH); plt.show()

        print("Graphs saved to:", self.config.MODEL_DIR)


## 10. Run it — load existing model automatically, or train a new one (>=100k episodes)

In [ ]:
# Optional: fix a random seed for reproducibility (bonus feature). Leave commented for full randomness.
# Config.set_seed(42)

manager = ModelManager(Config)
agent, history, trainer = manager.get_or_train()   # loads maze_agent.zip automatically if present


## 11. Training graphs

In [ ]:
visualizer = Visualizer(Config)
visualizer.plot_training_graphs(history)


## 12. Test on 20 brand-new, never-seen-during-training mazes

In [ ]:
tester = Tester(agent, Config)
test_results = tester.run_test_suite(n_mazes=Config.N_TEST_MAZES)


## 13. Final demo — one brand-new maze, animated step-by-step, with full report

In [ ]:
final_generator = MazeGenerator()
final_grid, final_start, final_goal = final_generator.generate()

print("Final demo maze:")
for line in final_generator.render_grid_with_symbols(final_grid, final_start, final_goal):
    print(line)

start_time = time.time()
final_result = tester.run_single_test(grid=final_grid, start=final_start, goal=final_goal, render=False)
time_taken = time.time() - start_time

print(f"Success: {final_result['success']}")
print(f"Total Reward: {final_result['reward']:.2f}")
print(f"Total Steps: {final_result['steps']}")
print(f"Time Taken: {time_taken:.4f} seconds")

visualizer.animate_episode(final_result, episode_label="Final Demo")


In [ ]:
print("=" * 50)
print("Training Complete")
print("Model Saved Successfully")
print(f"Model location: {Config.MODEL_PATH}")
print("=" * 50)


---
### Notes
- **Training time**: 100,000 episodes (~4M timesteps) of PPO on a Colab GPU/CPU runtime can take a
  long time. If you just want to verify the pipeline works end-to-end, temporarily lower
  `Config.TOTAL_EPISODES` (e.g. to 2,000) before running the training cell, then raise it back up
  for a real training run.
- **Resuming training**: call `manager.resume_training(extra_timesteps=1_000_000)` to continue
  training an existing `maze_agent.zip`.
- **TensorBoard**: logs are written to `Config.TENSORBOARD_DIR`. In Colab run:
  `%load_ext tensorboard` then `%tensorboard --logdir {Config.TENSORBOARD_DIR}`.
- **GPU**: automatically used if available (`Config.DEVICE`); for this MLP-based policy GPU speedup
  is modest since the network is small, but it is honored if present.
